In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from detanet_model.detanet_pbc import *
from detanet_model.metrics import *
from torch_geometric.loader import DataLoader
from e3nn import o3
import os
from torch_geometric.data import Data
device=torch.device('cuda')#cuda:0
dtype=torch.float32


In [ ]:
model=DetaNet(num_features=128,
                 act='swish',
                 maxl=3,
                 num_block=3,
                 radial_type='trainable_bessel',
                 num_radial=64,
                 attention_head=8,
                 cutoff_lower=0.0,
                 cutoff_upper=4,
                 max_num_neighbors=120,
                 strategy="brute",
                 check_errors=True,
                 box_vecs=None,
                 dropout=0.0,
                 use_cutoff=False,
                 max_atomic_number=35,
                 atom_ref=None,
                 scale=None,
                 scalar_outsize=1,
                 irreps_out=None,
                 summation=True,
                 norm=False,
                 out_type='scalar',
                 grad_type='ef',
                 device=device)


In [ ]:
#model.load_state_dict(torch.load('/root/shared-nvme/trained_param/qme14s/ef.pth'))

In [ ]:
import os
import torch
from torch import nn


class Trainer:
    def __init__(self, model, train_loader, val_loader=None,
                 loss_function=None,
                 device=torch.device('cuda:0'),
                 optimizer='Adam_amsgrad', lr=5e-4, weight_decay=0):
        self.opt_type = optimizer
        self.device = device
        self.model = model.to(device)
        self.train_data = train_loader
        self.val_data = val_loader
        self.loss_function = loss_function or nn.MSELoss()
        self.step = -1

        
        self.opts = {
            'AdamW': torch.optim.AdamW(self.model.parameters(), lr=lr, amsgrad=False, weight_decay=weight_decay),
            'AdamW_amsgrad': torch.optim.AdamW(self.model.parameters(), lr=lr, amsgrad=True, weight_decay=weight_decay),
            'Adam': torch.optim.Adam(self.model.parameters(), lr=lr, amsgrad=False, weight_decay=weight_decay),
            'Adam_amsgrad': torch.optim.Adam(self.model.parameters(), lr=lr, amsgrad=True, weight_decay=weight_decay),
            'Adadelta': torch.optim.Adadelta(self.model.parameters(), lr=lr, weight_decay=weight_decay),
            'RMSprop': torch.optim.RMSprop(self.model.parameters(), lr=lr, weight_decay=weight_decay),
            'SGD': torch.optim.SGD(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        }
        self.optimizer = self.opts[self.opt_type]

   
    def train(self, num_epoch, targ, stop_loss=1e-8, loss_area=[100000, 1e-8, 1e8],
              val_per_train=10, view_data=False, print_per_epoch=10, data_scale=1):
        self.model.train()
        len_train = len(self.train_data)

        for i in range(num_epoch):
            val_iter = iter(self.val_data) if self.val_data is not None else None
            for j, batch in enumerate(self.train_data):
                try:
                    self.step += 1
                    torch.cuda.empty_cache()
                    self.optimizer.zero_grad()

                    pos = batch.pos.to(self.device, dtype=torch.float32)
                    out = self.model(pos=pos, z=batch.z.to(self.device),
                                     batch=batch.batch.to(self.device), box=batch.box.to(self.device))
                    target = batch[targ].to(self.device) * data_scale

                    loss = self.loss_function(out.reshape(target.shape), target)

                    
                    if self.step < loss_area[0] or (loss_area[1] < loss.item() < loss_area[2]):
                        loss.backward()
                        self.optimizer.step()

                    # 验证
                    if (self.step % val_per_train == 0) and (self.val_data is not None):
                        val_batch = next(val_iter)
                        val_pos = val_batch.pos.to(self.device, dtype=torch.float32)
                        val_target = val_batch[targ].to(self.device).reshape(-1) * data_scale

                        with torch.no_grad():
                            val_out = self.model(pos=val_pos, z=val_batch.z.to(self.device),
                                                 batch=val_batch.batch.to(self.device), box=val_batch.box.to(self.device)
                                                 ).reshape(val_target.shape)
                            val_loss = self.loss_function(val_out, val_target).item()
                            val_mae = l1loss(val_out, val_target).item()
                            val_R2_val = R2(val_out, val_target).item()

                        if self.step % print_per_epoch == 0:
                            print(f'Epoch[{self.step}/{num_epoch * len_train}], '
                                  f'loss:{loss.item():.8f}, val_loss:{val_loss:.8f}, '
                                  f'val_mae:{val_mae:.8f}, val_R2:{val_R2_val:.8f}')
                            if view_data:
                                print(f'valout:{val_out.flatten()[0].item():.8f}, '
                                      f'valtarget:{val_target.flatten()[0].item():.8f}')

                        
                        assert (loss.item() > stop_loss) or (val_loss > stop_loss), \
                            'Training and prediction Loss below threshold, stopping.'

                    elif (self.step % print_per_epoch == 0) and (self.step % val_per_train != 0):
                        print(f'Epoch[{self.step}/{num_epoch * len_train}], loss:{loss.item():.8f}')

                except Exception as e:
                    print(f'[Error at step {self.step}] {e}')

                
                if self.step % 1000 == 0:
                    self.save_param(f'trained_param/time_save/param_{self.step}.pth')

   
    def train_ef_combine(self, num_epoch, loss_area=[100000, 1e-8, 1e8],
                         targ_e='atomization_energy', targ_f='force', stop_loss=0.0,
                         val_per_train=10, view_data=False, print_per_epoch=10,
                         loss_function=None, coff=1000):
        self.model.train()
        len_train = len(self.train_data)
        loss_function = loss_function or ef_combine_loss

        for i in range(num_epoch):
            val_iter = iter(self.val_data) if self.val_data is not None else None
            for j, batch in enumerate(self.train_data):
                try:
                    batch = batch.to(self.device)
                    self.step += 1
                    torch.cuda.empty_cache()
                    self.optimizer.zero_grad()

                    e_pred, f_pred = self.model(pos=batch.pos, z=batch.z,
                                                batch=batch.batch, box=batch.box)
                    e_targ = batch[targ_e].reshape(e_pred.shape)
                    f_targ = batch[targ_f].reshape(f_pred.shape)

                    loss = loss_function(e_pred, e_targ, f_pred, f_targ, coff)

                    if self.step < loss_area[0] or (loss_area[1] < loss.item() < loss_area[2]):
                        loss.backward()
                        self.optimizer.step()

                    if (self.step % val_per_train == 0) and (self.val_data is not None):
                        val_batch = next(val_iter).to(self.device)
                        val_e_pred, val_f_pred = self.model(pos=val_batch.pos, z=val_batch.z,
                                                            batch=val_batch.batch, box=val_batch.box)
                        val_e_targ = val_batch[targ_e].reshape(val_e_pred.shape)
                        val_f_targ = val_batch[targ_f].reshape(val_f_pred.shape)
                        with torch.no_grad():
                            val_loss = loss_function(val_e_pred, val_e_targ, val_f_pred, val_f_targ, coff).item()
                            val_mae_e = l1loss(val_e_pred, val_e_targ).item()
                            val_mae_f = l1loss(val_f_pred, val_f_targ).item()
                            val_R2_e = R2(val_e_pred, val_e_targ).item()
                            val_R2_f = R2(val_f_pred, val_f_targ).item()

                        if self.step % print_per_epoch == 0:
                            print(f'Epoch[{self.step}/{num_epoch * len_train}], '
                                  f'loss:{loss.item():.8f}, val_loss:{val_loss:.8f}, '
                                  f'e_mae:{val_mae_e:.6f}, e_r2:{val_R2_e:.6f}, '
                                  f'f_mae:{val_mae_f:.6f}, f_r2:{val_R2_f:.6f}')

                        assert (loss.item() > stop_loss) or (val_loss > stop_loss), \
                            'Training and prediction Loss below threshold, stopping.'

                    elif (self.step % print_per_epoch == 0) and (self.step % val_per_train != 0):
                        print(f'Epoch[{self.step}/{num_epoch * len_train}], loss:{loss.item():.8f}')

                except Exception as e:
                    print(f'[Error at step {self.step}] {e}')

                if self.step % 1800 == 0:
                    self.save_param(f'trained_param/time_save/param_{self.step}.pth')

   
    def _ensure_dir(self, path):
        directory = os.path.dirname(path)
        if directory and not os.path.exists(directory):
            os.makedirs(directory, exist_ok=True)

    def save_param(self, path):
        self._ensure_dir(path)
        torch.save(self.model.state_dict(), path)

    def save_model(self, path):
        self._ensure_dir(path)
        torch.save(self.model, path)

    def save_opt(self, path):
        self._ensure_dir(path)
        torch.save(self.optimizer.state_dict(), path)

    def load_state_and_optimizer(self, state_path=None, optimizer_path=None):
        if state_path is not None:
            state_dict = torch.load(state_path, map_location=self.device)
            self.model.load_state_dict(state_dict)
        if optimizer_path is not None:
            opt_state = torch.load(optimizer_path, map_location=self.device)
            self.optimizer.load_state_dict(opt_state)

    def params(self):
        return self.model.state_dict()


In [ ]:
model=model.to(dtype)
model=model.to(device)

In [ ]:
'''
datasets=torch.load('/root/shared-nvme/data/NEP-MB-pol/MB-pol-104607.pt', weights_only=False)
train_datasets=[]
val_datasets1=[]
for i in range(len(datasets)):
    zerr=0
    data=datasets[i]
    data_=Data(pos=data.pos,z=data.z,box=data.box,force=data.force,energy=data.energy)
    if zerr==0:
        if i%(10*1)==0:
            val_datasets1.append(data_)
        else:
            train_datasets.append(data_)
    else:
        print('error',data.smile)
print(len(train_datasets))
print(train_datasets[0])
print(len(val_datasets1))
print(val_datasets1[0])
'''

In [ ]:
#train_datasets=torch.load('/root/shared-nvme/data/water_force_energy-training.pt', weights_only=False)

In [ ]:
#val_datasets=torch.load('/root/shared-nvme/data/water_force_energy-validation.pt', weights_only=False)

In [ ]:

val_datasets=[]
test_datasets = []
for i in range(len(val_datasets1)):
    d = val_datasets1[i]
    if i % (2*1) == 0:
        val_datasets.append(d)
    else:
        test_datasets.append(d)
print(len(val_datasets))
print(len(test_datasets))


In [ ]:
bathes=64
trainloader=DataLoader(train_datasets,batch_size=bathes,shuffle=True)
valloader=DataLoader(val_datasets,batch_size=bathes,shuffle=True)

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=1e-3,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef1.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef1.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=4e-4,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef2.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef2.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=2e-4,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef3.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef3.pth'))


In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=1e-4,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef4.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef4.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=4e-5,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef5.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef5.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=2e-5,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch= 15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef6.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef6.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=1e-5,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch=15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef7.pth')

In [ ]:
model.load_state_dict(torch.load('trained_param/ef7.pth'))

In [ ]:
trainer=Trainer(model,train_loader=trainloader,val_loader=valloader,loss_function=l2loss,lr=8e-6,weight_decay=0,optimizer='AdamW',device=device)

In [ ]:
trainer.train_ef_combine(num_epoch=15, loss_area=[150,1e-8,0.1],targ_e='energy',targ_f='force',    #atomization_
                         val_per_train=150,view_data=False,print_per_epoch=30, loss_function=ef_combine_loss, coff=1)

In [ ]:
trainer.save_param('trained_param/ef8.pth')